# پروژه دوم: پیش‌بینی قیمت خانه با CNN یک‌بعدی
**محیط اجرا:** Google Colab  |  **زبان:** Python
این نوت‌بوک را از ابتدا تا انتها اجرا کنید. برای دریافت دیتاست به حساب Kaggle و فایل `kaggle.json` نیاز است.

In [ ]:
# پروژه دوم: پیش‌بینی قیمت خانه با شبکه CNN یک‌بعدی
# Kaggle Competition: House Prices - Advanced Regression Techniques
import os, pathlib, shutil, random
import numpy as np, pandas as pd, tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 1) دریافت خودکار دیتاست بدون kaggle.json
# فایل‌های CSV از یک مخزن عمومی GitHub که نسخه‌ای از داده‌های مسابقه را نگهداری می‌کند دریافت می‌شوند.
import urllib.request
data_dir="/content/house_data"
os.makedirs(data_dir,exist_ok=True)
base_url="https://raw.githubusercontent.com/Rswcf/houseprices_kaggle/master/"
for filename in ["train.csv","test.csv"]:
    url=base_url+filename
    destination=os.path.join(data_dir,filename)
    print("Downloading",filename)
    urllib.request.urlretrieve(url,destination)
    print(filename,"downloaded:",os.path.getsize(destination),"bytes")
train_path=os.path.join(data_dir,"train.csv")
test_path=os.path.join(data_dir,"test.csv")
train=pd.read_csv(train_path)
kaggle_test=pd.read_csv(test_path)
print("Train:",train.shape,"Test:",kaggle_test.shape)
display(train.head())
print("Dataset downloaded automatically; no kaggle.json required.")

In [ ]:
# 2) شناخت داده و جداسازی متغیر هدف
TARGET="SalePrice"
print(train[TARGET].describe())
print("Missing values (top 15):")
display(train.isna().sum().sort_values(ascending=False).head(15))
y=np.log1p(train[TARGET].astype(float))  # تبدیل لگاریتمی برای کاهش چولگی قیمت
X=train.drop(columns=[TARGET, "Id"], errors="ignore")
X_kaggle=kaggle_test.drop(columns=["Id"], errors="ignore")
# ستون‌ها را هم‌تراز می‌کنیم تا ویژگی‌های train و test یکسان باشند.
X_kaggle=X_kaggle.reindex(columns=X.columns)
numeric_cols=X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols=X.select_dtypes(exclude=["number"]).columns.tolist()
print("Numeric features:", len(numeric_cols), "| Categorical:", len(categorical_cols))

In [ ]:
# 3) پیش‌پردازش: جایگزینی مقادیر گمشده، مقیاس‌بندی و One-Hot Encoding
# fit فقط روی بخش آموزش انجام می‌شود تا نشت اطلاعات رخ ندهد.
numeric_pipe=Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe=Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor=ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols)
], remainder="drop")

X_train, X_valid, y_train, y_valid=train_test_split(
    X, y, test_size=0.20, random_state=SEED
)
X_train_p=preprocessor.fit_transform(X_train).astype("float32")
X_valid_p=preprocessor.transform(X_valid).astype("float32")
print("Processed shape:", X_train_p.shape, X_valid_p.shape)

In [ ]:
# 4) طراحی CNN یک‌بعدی برای ویژگی‌های جدولی
# ویژگی‌های پیش‌پردازش‌شده مانند یک توالی یک‌بعدی به Conv1D داده می‌شوند.
# این انتخاب برای اجرای خواسته پروژه است؛ برای داده جدولی مدل‌های Dense/درختی نیز معیار مقایسه‌اند.
from tensorflow.keras import layers, models
n_features=X_train_p.shape[1]
model=models.Sequential([
    layers.Input(shape=(n_features,1)),
    layers.Conv1D(32, kernel_size=3, activation="relu", padding="same"),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(64, kernel_size=3, activation="relu", padding="same"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.25),
    layers.Dense(1)
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

In [ ]:
# 5) آموزش و رسم نمودار MSE برحسب Epoch
callbacks=[
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("/content/house_cnn_best.keras", save_best_only=True)
]
history=model.fit(
    X_train_p[...,None], y_train.to_numpy(),
    validation_data=(X_valid_p[...,None], y_valid.to_numpy()),
    epochs=150, batch_size=32, callbacks=callbacks, verbose=1
)
plt.figure(figsize=(7,4))
plt.plot(history.history["loss"], label="Train MSE")
plt.plot(history.history["val_loss"], label="Validation MSE")
plt.xlabel("Epoch"); plt.ylabel("MSE (log price)"); plt.title("MSE per Epoch")
plt.legend(); plt.grid(True); plt.show()

In [ ]:
# 6) ارزیابی رگرسیون روی validation
# معیارهای مناسب رگرسیون: MSE, RMSE, MAE, R2
pred_log=model.predict(X_valid_p[...,None]).reshape(-1)
true_price=np.expm1(y_valid.to_numpy())
pred_price=np.maximum(0, np.expm1(pred_log))
mse=mean_squared_error(true_price,pred_price)
rmse=np.sqrt(mse)
mae=mean_absolute_error(true_price,pred_price)
r2=r2_score(true_price,pred_price)
print(f"MSE : {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"MAE : {mae:,.2f}")
print(f"R2  : {r2:.4f}")

plt.figure(figsize=(6,6))
plt.scatter(true_price,pred_price,alpha=.6)
lims=[min(true_price.min(),pred_price.min()), max(true_price.max(),pred_price.max())]
plt.plot(lims,lims,"r--"); plt.xlabel("Actual price"); plt.ylabel("Predicted price")
plt.title("Actual vs Predicted"); plt.grid(True); plt.show()

In [ ]:
# 7) معیارهای طبقه‌بندی به‌صورت تحلیل تکمیلیِ گروه‌های قیمتی
# این معیارها ذاتاً برای رگرسیون نیستند. برای گزارش تکمیلی، قیمت را به دو گروه
# «پایین‌تر/بالاتر از میانه مجموعه آموزش» تبدیل می‌کنیم.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
threshold=float(np.expm1(y_train.median()))
actual_group=(true_price >= threshold).astype(int)
pred_group=(pred_price >= threshold).astype(int)
print("Median price threshold:", round(threshold,2))
print("Accuracy :", accuracy_score(actual_group,pred_group))
print("Precision:", precision_score(actual_group,pred_group,zero_division=0))
print("Recall   :", recall_score(actual_group,pred_group,zero_division=0))
print("F1-score :", f1_score(actual_group,pred_group,zero_division=0))
cm=confusion_matrix(actual_group,pred_group)
ConfusionMatrixDisplay(cm,display_labels=["Below median","At/above median"]).plot(cmap="Blues",values_format="d")
plt.title("Confusion Matrix - Price groups (supplementary)"); plt.show()

In [ ]:
# 8) بررسی اثر لایه‌ها: مقایسه معماری‌های کم‌عمق و عمیق‌تر
def build_cnn(depth=1):
    inp=layers.Input(shape=(n_features,1))
    x=inp
    filters=[32,64,96][:depth]
    for f in filters:
        x=layers.Conv1D(f,3,padding="same",activation="relu")(x)
        if int(x.shape[1]) > 2:
            x=layers.MaxPooling1D(2)(x)
    x=layers.GlobalAveragePooling1D()(x)
    x=layers.Dense(64,activation="relu")(x)
    out=layers.Dense(1)(x)
    m=models.Model(inp,out)
    m.compile(optimizer="adam",loss="mse",metrics=["mae"])
    return m

depth_results={}
for depth in [1,2,3]:
    m=build_cnn(depth)
    m.fit(X_train_p[...,None],y_train.to_numpy(),validation_data=(X_valid_p[...,None],y_valid.to_numpy()),
          epochs=35,batch_size=32,verbose=0,
          callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=6,restore_best_weights=True)])
    p=np.expm1(m.predict(X_valid_p[...,None],verbose=0).reshape(-1))
    depth_results[f"{depth} Conv1D layer(s)"]=np.sqrt(mean_squared_error(true_price,p))
print("Validation RMSE by depth:",depth_results)
plt.figure(figsize=(7,4)); plt.bar(depth_results.keys(),depth_results.values())
plt.ylabel("Validation RMSE"); plt.title("Effect of CNN depth (lower is better)")
plt.xticks(rotation=15); plt.grid(axis="y"); plt.show()

In [ ]:
# 9) پیش‌بینی مجموعه تست مسابقه و ساخت فایل submission.csv
# Kaggle test فاقد SalePrice است؛ بنابراین ارزیابی نهایی روی validation انجام شد.
# این سلول صرفاً پیش‌بینی‌های موردنیاز فایل ارسال مسابقه را می‌سازد.
X_kaggle_p=preprocessor.transform(X_kaggle).astype("float32")
test_pred_log=model.predict(X_kaggle_p[...,None]).reshape(-1)
test_pred=np.maximum(0,np.expm1(test_pred_log))
submission=pd.DataFrame({"Id":kaggle_test["Id"],"SalePrice":test_pred})
submission.to_csv("/content/submission.csv",index=False)
model.save("/content/house_price_cnn.keras")
display(submission.head())
print("Created /content/submission.csv and /content/house_price_cnn.keras")
from google.colab import files
files.download("/content/submission.csv")